# Preprocessing and Feature Engineering

This stage prepares the Obfuscated-MalMem2022 dataset for later model training. The test set is created before any preprocessing is fitted, and no model is trained here.

`Category` is inspected and excluded because it has very high cardinality and contains descriptive sample/category text rather than a compact behavioral feature. Keeping it would create a large, brittle representation and could reduce generalization to unseen malware samples.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
dataset_path = project_root / 'dataset' / 'Obfuscated-MalMem2022.csv'
models_path = project_root / 'models'
models_path.mkdir(exist_ok=True)

df = pd.read_csv(dataset_path)
target_column = 'Class'
category_column = 'Category'

print(f'Dataset path: {dataset_path}')
print(f'Original dataset shape: {df.shape}')

Dataset path: D:\aics pbl\dataset\Obfuscated-MalMem2022.csv
Original dataset shape: (58058, 57)


In [2]:
if target_column not in df.columns:
    raise KeyError(f'Missing target column: {target_column}')

X_all = df.drop(columns=[target_column])
y = df[target_column].copy()

category_summary = {
    'unique_values': int(df[category_column].nunique(dropna=False)),
    'unique_ratio': float(df[category_column].nunique(dropna=False) / len(df)),
    'dtype': str(df[category_column].dtype),
}

print(f'Input features before removing unsuitable columns: {X_all.shape[1]}')
print(f"Category analysis: {category_summary['unique_values']:,} unique values ({category_summary['unique_ratio']:.2%} of rows), dtype={category_summary['dtype']}")
print('Target distribution:')
print(y.value_counts().to_string())

numeric_all = X_all.select_dtypes(include=np.number)
constant_columns = numeric_all.columns[numeric_all.nunique(dropna=False) <= 1].tolist()
duplicate_feature_columns = numeric_all.columns[numeric_all.T.duplicated()].tolist()

removed_columns = {
    category_column: 'Excluded: high-cardinality descriptive text would create a large, brittle representation and is not a compact behavioral feature.',
    **{column: 'Excluded: constant/zero-variance feature; it carries no predictive information.' for column in constant_columns},
}
X = X_all.drop(columns=list(removed_columns))
print(f'Input features after removing unsuitable columns: {X.shape[1]}')
print(f'Columns removed: {list(removed_columns)}')
print(f'Duplicate numeric feature columns found before removal: {duplicate_feature_columns}')

Input features before removing unsuitable columns: 56
Category analysis: 28,346 unique values (48.82% of rows), dtype=object
Target distribution:
Class
Benign     29227
Malware    28831


Input features after removing unsuitable columns: 52
Columns removed: ['Category', 'pslist.nprocs64bit', 'handles.nport', 'svcscan.interactive_process_services']
Duplicate numeric feature columns found before removal: ['handles.nport', 'svcscan.interactive_process_services']


In [3]:
non_numeric_columns = X.select_dtypes(exclude=np.number).columns.tolist()
if non_numeric_columns:
    raise TypeError(f'Remaining features must be numeric; found: {non_numeric_columns}')

infinite_value_count = int((~np.isfinite(X.to_numpy(dtype=float))).sum())
missing_value_count = int(X.isna().sum().sum())
constant_columns_after_removal = X.columns[X.nunique(dropna=False) <= 1].tolist()
duplicate_feature_columns_after_removal = X.columns[X.T.duplicated()].tolist()

print(f'Remaining non-numeric columns: {non_numeric_columns}')
print(f'NaN values in features: {missing_value_count}')
print(f'Infinite values in features: {infinite_value_count}')
print(f'Constant/zero-variance columns found before removal: {constant_columns}')
print(f'Duplicate feature columns found before removal: {duplicate_feature_columns}')
print(f'Constant/zero-variance columns after removal: {constant_columns_after_removal}')
print(f'Duplicate feature columns after removal: {duplicate_feature_columns_after_removal}')

if infinite_value_count:
    raise ValueError('Infinite feature values must be handled before training.')
if constant_columns_after_removal:
    raise ValueError(f'Constant columns remain: {constant_columns_after_removal}')
if duplicate_feature_columns_after_removal:
    raise ValueError(f'Duplicate feature columns remain: {duplicate_feature_columns_after_removal}')

Remaining non-numeric columns: []
NaN values in features: 0
Infinite values in features: 0
Constant/zero-variance columns found before removal: ['pslist.nprocs64bit', 'handles.nport', 'svcscan.interactive_process_services']
Duplicate feature columns found before removal: ['handles.nport', 'svcscan.interactive_process_services']
Constant/zero-variance columns after removal: []
Duplicate feature columns after removal: []


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

numeric_features = X.columns.tolist()
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)
tree_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
        ]), numeric_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

# Fit only on the training split; the test split is transformed but never used to fit.
X_train_scaled = scaled_preprocessor.fit_transform(X_train)
X_test_scaled = scaled_preprocessor.transform(X_test)
X_train_tree = tree_preprocessor.fit_transform(X_train)
X_test_tree = tree_preprocessor.transform(X_test)

print(f'Training set shape: {X_train.shape}')
print(f'Testing set shape: {X_test.shape}')
print('Training class distribution:')
print(y_train.value_counts().to_string())
print('Testing class distribution:')
print(y_test.value_counts().to_string())
print(f'Scaled training matrix shape: {X_train_scaled.shape}')
print(f'Tree-model training matrix shape: {X_train_tree.shape}')
print('Preprocessing: median imputation is configured; StandardScaler is used only by scaled_preprocessor.')

Training set shape: (46446, 52)
Testing set shape: (11612, 52)
Training class distribution:
Class
Benign     23381
Malware    23065
Testing class distribution:
Class
Benign     5846
Malware    5766
Scaled training matrix shape: (46446, 52)
Tree-model training matrix shape: (46446, 52)
Preprocessing: median imputation is configured; StandardScaler is used only by scaled_preprocessor.


In [5]:
preprocessing_config = {
    'dataset_path': str(dataset_path),
    'target_column': target_column,
    'feature_columns': numeric_features,
    'removed_columns': removed_columns,
    'original_shape': list(df.shape),
    'feature_count_before_removal': int(X_all.shape[1]),
    'feature_count_after_removal': int(X.shape[1]),
    'test_size': 0.20,
    'random_state': 42,
    'stratified_split': True,
    'scaled_preprocessing': ['median imputation', 'StandardScaler'],
    'tree_preprocessing': ['median imputation'],
}

with (models_path / 'preprocessing_config.json').open('w', encoding='utf-8') as file:
    json.dump(preprocessing_config, file, indent=2)

joblib.dump(
    {
        'scaled_preprocessor': scaled_preprocessor,
        'tree_preprocessor': tree_preprocessor,
        'feature_columns': numeric_features,
        'target_classes': sorted(y.unique().tolist()),
    },
    models_path / 'preprocessing_artifacts.joblib',
)

print('Saved models/preprocessing_config.json')
print('Saved models/preprocessing_artifacts.joblib')
print('No model was trained in this preprocessing stage.')

Saved models/preprocessing_config.json
Saved models/preprocessing_artifacts.joblib
No model was trained in this preprocessing stage.
